In [2]:
# %%
"""
=========================================================================================
PROJECT: LANDSLIDE FORECASTING - Bi-LSTM ULTIMATE (Balanced)
=========================================================================================
[MODEL] Bidirectional LSTM
[DATA STRATEGY]
- Multi-Horizon Forecast (0m - 24h)
- Explicit Data Balancing (Oversampling Minority + Downsampling Majority)
"""

import os
import glob
import time
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

import warnings
warnings.filterwarnings('ignore')

# ====================================================================
# 1. CONFIGURATION
# ====================================================================
class Config:
    DATA_DIR = "./data/"
    MODEL_DIR = "./model/bilstm_balanced/"
    SCALER_PATH = "./model/bilstm_bal_scaler.save"
    
    HORIZONS = [0, 30, 60, 180, 360, 720, 1440]
    SEQUENCE_LENGTH = 60
    TEST_KEYWORDS = ['label_for_dev109_test_prepared']
    
    RAW_COLS = ['rain', 'soil', 'temp', 'humi', 'geo']
    LABEL_COL = 'label'
    LABEL_MAP = {'normal': 0, 'warning': 1, 'critical': 2}
    
    # --- BALANCING TARGET ---
    # ปรับให้ทุกคลาสมีจำนวนเท่านี้ (Normal จะถูกตัดออก, Critical จะถูกปั๊มเพิ่ม)
    TARGET_SAMPLES = 5000 
    
    # Training
    BATCH_SIZE = 64
    EPOCHS = 100
    LEARNING_RATE = 1e-3
    
    # Safety Weights (ใส่ไว้นิดหน่อยเพื่อเน้นย้ำ แม้จะ Balance แล้ว)
    CLASS_WEIGHTS = {0: 1.0, 1: 10.0, 2: 20.0}

cfg = Config()
if not os.path.exists(cfg.MODEL_DIR): os.makedirs(cfg.MODEL_DIR)
np.random.seed(42); tf.random.set_seed(42); random.seed(42)

def log(msg): print(f"[INFO] {time.strftime('%H:%M:%S')} - {msg}")
print("✅ Bi-LSTM Config Loaded.")

# ====================================================================
# 2. LOAD & FEATURE ENGINEERING
# ====================================================================
def load_data():
    log("Scanning Data...")
    files = glob.glob(os.path.join(cfg.DATA_DIR, "*.csv"))
    if not files: files = glob.glob(os.path.join(cfg.DATA_DIR, "**/*.csv"), recursive=True)
    
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f)
            df.columns = [str(c).lower().strip().replace('.1', '') for c in df.columns]
            rename_map = {'temperature':'temp', 'hum':'humi', 'humidity':'humi', 'devid':'devID', 'time':'timestamp', 'date':'timestamp'}
            new_cols = {c: rename_map[c] for c in df.columns if c in rename_map}
            if new_cols: df = df.rename(columns=new_cols)
            
            if 'devID' in df.columns: df['devID'] = df['devID'].astype(str).str.extract('(\d+)')[0].astype(float).fillna(0).astype(int)
            if 'timestamp' in df.columns: df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
            for c in cfg.RAW_COLS: 
                if c not in df.columns: df[c] = 0.0
            
            filename = os.path.basename(f)
            df['is_test'] = any(k in filename for k in cfg.TEST_KEYWORDS)
            dfs.append(df)
        except: pass
    return pd.concat(dfs, ignore_index=True) if dfs else None

def generate_features(df):
    log("Generating Features...")
    df = df.sort_values(['devID', 'timestamp']).reset_index(drop=True)
    df_list = []
    
    for dev, g in df.groupby('devID'):
        g = g.set_index('timestamp')
        g = g[~g.index.duplicated(keep='first')]
        if len(g) > 0: g = g.resample('1T').asfreq()
        g[cfg.RAW_COLS] = g[cfg.RAW_COLS].interpolate(limit_direction='both').fillna(0)
        
        # Physics Features
        g['rain_1h'] = g['rain'].rolling(60, min_periods=1).sum()
        g['soil_rate'] = g['soil'].diff().fillna(0)
        g['soil_acc']  = g['soil'].diff().diff().fillna(0)
        g['rain_6h'] = g['rain'].rolling(360, min_periods=1).sum()
        g['soil_trend_6h'] = g['soil'] - g['soil'].shift(360).fillna(method='bfill')
        g['rain_24h'] = g['rain'].rolling(1440, min_periods=1).sum()
        g['soil_max_24h'] = g['soil'].rolling(1440, min_periods=1).max()
        g['rain_x_soil'] = g['rain'] * g['soil']
        g['rain_x_geo']  = g['rain'] * g['geo'].abs()
        
        if cfg.LABEL_COL in g.columns:
            g[cfg.LABEL_COL] = g[cfg.LABEL_COL].fillna('normal').astype(str).str.lower().str.strip()
            g[cfg.LABEL_COL] = g[cfg.LABEL_COL].map(cfg.LABEL_MAP).fillna(0).astype(int)
        else: g[cfg.LABEL_COL] = 0
        
        g['is_test'] = g['is_test'].fillna(method='ffill').fillna(method='bfill')
        g['devID'] = dev
        df_list.append(g.reset_index())
        
    return pd.concat(df_list, ignore_index=True).fillna(0)

df_train_raw = load_data()
df_proc = generate_features(df_train_raw)

# ====================================================================
# 3. SEQUENCE & BALANCING
# ====================================================================
def create_sequences(df, feature_cols, horizon):
    Xs, ys, is_tests = [], [], []
    for dev, g in df.groupby('devID'):
        data = g[feature_cols].values
        labels = g[cfg.LABEL_COL].shift(-horizon).values
        test_flag = g['is_test'].values
        
        valid_len = len(g) - horizon
        if valid_len < cfg.SEQUENCE_LENGTH: continue
        
        for i in range(valid_len - cfg.SEQUENCE_LENGTH):
            Xs.append(data[i : i+cfg.SEQUENCE_LENGTH])
            ys.append(labels[i+cfg.SEQUENCE_LENGTH])
            is_tests.append(test_flag[i+cfg.SEQUENCE_LENGTH])
            
    return np.array(Xs), np.array(ys), np.array(is_tests)

def balance_data(X, y):
    """
    ฟังก์ชันปรับสมดุลข้อมูล:
    - ถ้ามีเยอะ -> สุ่มออก (Downsample)
    - ถ้ามีน้อย -> สุ่มเพิ่ม (Oversample)
    """
    print(f"   Original counts: {Counter(y)}")
    X_bal, y_bal = [], []
    
    for cls in np.unique(y):
        if np.isnan(cls): continue
        indices = np.where(y == cls)[0]
        count = len(indices)
        
        # ตัดสินใจว่าจะเพิ่มหรือลด
        replace = count < cfg.TARGET_SAMPLES
        
        chosen_indices = np.random.choice(
            indices, 
            cfg.TARGET_SAMPLES, 
            replace=replace
        )
        
        X_bal.append(X[chosen_indices])
        y_bal.append(y[chosen_indices])
        
    X_bal = np.concatenate(X_bal)
    y_bal = np.concatenate(y_bal)
    
    # Shuffle เพื่อไม่ให้ข้อมูลเรียงกันเป็นก้อนๆ
    perm = np.random.permutation(len(X_bal))
    return X_bal[perm], y_bal[perm]

# ====================================================================
# 4. Bi-LSTM MODEL ARCHITECTURE
# ====================================================================
def build_bilstm_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    
    # Bi-LSTM 1: จับ Pattern ซับซ้อน
    x = Bidirectional(LSTM(128, return_sequences=True))(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    # Bi-LSTM 2: สรุปผล
    x = Bidirectional(LSTM(64, return_sequences=False))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    # Dense Layer
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    
    # Output
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name="BiLSTM_Forecaster")
    model.compile(optimizer=Adam(learning_rate=cfg.LEARNING_RATE), 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

# ====================================================================
# 5. TRAINING LOOP
# ====================================================================
def train_system():
    FINAL_FEATURES = cfg.RAW_COLS + ['rain_1h', 'soil_rate', 'soil_acc', 'rain_6h', 'soil_trend_6h', 'rain_24h', 'soil_max_24h', 'rain_x_soil', 'rain_x_geo']
    
    log("Scaling Data...")
    scaler = RobustScaler()
    train_mask = ~df_proc['is_test']
    scaler.fit(df_proc.loc[train_mask, FINAL_FEATURES])
    df_proc[FINAL_FEATURES] = scaler.transform(df_proc[FINAL_FEATURES])
    joblib.dump(scaler, cfg.SCALER_PATH)
    
    results_log = []

    for h in cfg.HORIZONS:
        print(f"\n{'='*60}")
        print(f"🚀 Bi-LSTM HORIZON: +{h/60:.1f} Hours ({h} mins)")
        print(f"{'='*60}")
        
        # 1. Sequence Creation
        X_all, y_all, test_mask = create_sequences(df_proc, FINAL_FEATURES, h)
        
        # 2. Split Data
        X_train = X_all[~test_mask]; y_train = y_all[~test_mask]
        X_test  = X_all[test_mask];  y_test  = y_all[test_mask]
        
        valid_tr = ~np.isnan(y_train); X_train = X_train[valid_tr]; y_train = y_train[valid_tr]
        valid_ts = ~np.isnan(y_test);  X_test = X_test[valid_ts];   y_test = y_test[valid_ts]
        
        # Split Val
        X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
        
        # 3. BALANCE TRAIN
        print("⚖️ Balancing Train Data...")
        X_train_bal, y_train_bal = balance_data(X_train, y_train)
        print(f"   Balanced Counts: {Counter(y_train_bal)}")
        
        # Encode
        y_train_cat = to_categorical(y_train_bal, 3)
        y_val_cat = to_categorical(y_val, 3)
        
        # Weights
        safety_factor = 1.0 + (h/1440.0)
        cw = {k: v * safety_factor for k, v in cfg.CLASS_WEIGHTS.items()}
        
        # 4. Train
        model = build_bilstm_model((cfg.SEQUENCE_LENGTH, len(FINAL_FEATURES)), 3)
        save_path = os.path.join(cfg.MODEL_DIR, f"bilstm_h{h}.h5")
        
        cbs = [
            EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
            ReduceLROnPlateau(patience=5, factor=0.5, monitor='val_loss'),
            ModelCheckpoint(save_path, save_best_only=True, monitor='val_accuracy', verbose=0)
        ]
        
        model.fit(X_train_bal, y_train_cat, validation_data=(X_val, y_val_cat),
                  epochs=cfg.EPOCHS, batch_size=cfg.BATCH_SIZE, 
                  class_weight=cw, callbacks=cbs, verbose=1)
        
        # 5. Evaluate
        probs = model.predict(X_test, verbose=0)
        y_pred = np.argmax(probs, axis=1)
        
        report = classification_report(y_test, y_pred, target_names=['Normal', 'Warning', 'Critical'], output_dict=True, zero_division=0)
        f1_norm = report['Normal']['f1-score']
        f1_warn = report['Warning']['f1-score']
        f1_crit = report['Critical']['f1-score']
        acc = accuracy_score(y_test, y_pred)
        
        print(f"   📊 Accuracy : {acc:.4f}")
        print(f"   🟢 F1-Norm  : {f1_norm:.4f}")
        print(f"   🟡 F1-Warn  : {f1_warn:.4f}")
        print(f"   🔴 F1-Crit  : {f1_crit:.4f}")
        
        results_log.append({'Horizon': h, 'Acc': acc, 'F1_Norm': f1_norm, 'F1_Warn': f1_warn, 'F1_Crit': f1_crit})

    summary = pd.DataFrame(results_log)
    summary.to_csv("./model/bilstm_horizon_performance.csv", index=False)
    print("\n=== FINAL SUMMARY ===")
    print(summary)
    
    plt.figure(figsize=(10, 6))
    plt.plot(summary['Horizon'], summary['F1_Crit'], marker='o', color='red', label='Critical F1')
    plt.plot(summary['Horizon'], summary['F1_Warn'], marker='s', color='orange', label='Warning F1')
    plt.title("Bi-LSTM Forecast Performance")
    plt.xlabel("Minutes Ahead")
    plt.ylabel("F1 Score")
    plt.legend(); plt.grid(True)
    plt.show()

if __name__ == "__main__":
    train_system()

✅ Bi-LSTM Config Loaded.
[INFO] 02:18:25 - Scanning Data...
[INFO] 02:18:26 - Generating Features...
[INFO] 02:18:26 - Scaling Data...

🚀 Bi-LSTM HORIZON: +0.0 Hours (0 mins)
⚖️ Balancing Train Data...
   Original counts: Counter({np.int64(0): 24393, np.int64(1): 423, np.int64(2): 252})
   Balanced Counts: Counter({np.int64(2): 5000, np.int64(0): 5000, np.int64(1): 5000})
Epoch 1/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step - accuracy: 0.7084 - loss: 5.2209

235/235 ━━━━━━━━━━━━━━━━━━━━ 77s 265ms/step - accuracy: 0.7087 - loss: 5.2137 - val_accuracy: 0.6885 - val_loss: 0.8177 - learning_rate: 0.0010
Epoch 2/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - accuracy: 0.8356 - loss: 2.1719

235/235 ━━━━━━━━━━━━━━━━━━━━ 73s 223ms/step - accuracy: 0.8356 - loss: 2.1712 - val_accuracy: 0.7166 - val_loss: 0.8116 - learning_rate: 0.0010
Epoch 3/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step - accuracy: 0.8561 - loss: 1.7261

235/235 ━━━━━━━━━━━━━━━━━━━━ 51s 219ms/step - accuracy: 0.8562 - loss: 1.7257 - val_accuracy: 0.7305 - val_loss: 0.7882 - learning_rate: 0.0010
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - accuracy: 0.8721 - loss: 1.4112

235/235 ━━━━━━━━━━━━━━━━━━━━ 83s 220ms/step - accuracy: 0.8722 - loss: 1.4107 - val_accuracy: 0.7321 - val_loss: 0.7844 - learning_rate: 0.0010
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 0.8783 - loss: 1.2510

...

235/235 ━━━━━━━━━━━━━━━━━━━━ 86s 301ms/step - accuracy: 0.9652 - loss: 0.1839 - val_accuracy: 0.8801 - val_loss: 0.5298 - learning_rate: 2.5000e-04
Epoch 55/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 264ms/step - accuracy: 0.9627 - loss: 0.2043

235/235 ━━━━━━━━━━━━━━━━━━━━ 72s 305ms/step - accuracy: 0.9627 - loss: 0.2043 - val_accuracy: 0.8883 - val_loss: 0.5149 - learning_rate: 2.5000e-04
Epoch 56/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 77s 329ms/step - accuracy: 0.9668 - loss: 0.1959 - val_accuracy: 0.8802 - val_loss: 0.5268 - learning_rate: 2.5000e-04
Epoch 57/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 88s 351ms/step - accuracy: 0.9685 - loss: 0.1867 - val_accuracy: 0.8883 - val_loss: 0.5125 - learning_rate: 2.5000e-04
Epoch 58/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 154s 402ms/step - accuracy: 0.9683 - loss: 0.1986 - val_accuracy: 0.8843 - val_loss: 0.5077 - learning_rate: 2.5000e-04
Epoch 59/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 378ms/step - accuracy: 0.9703 - loss: 0.2046

235/235 ━━━━━━━━━━━━━━━━━━━━ 157s 467ms/step - accuracy: 0.9703 - loss: 0.2046 - val_accuracy: 0.8921 - val_loss: 0.4904 - learning_rate: 2.5000e-04
Epoch 60/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 138s 448ms/step - accuracy: 0.9717 - loss: 0.1679 - val_accuracy: 0.8817 - val_loss: 0.5348 - learning_rate: 2.5000e-04
Epoch 61/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 108s 462ms/step - accuracy: 0.9709 - loss: 0.1658 - val_accuracy: 0.8837 - val_loss: 0.5400 - learning_rate: 2.5000e-04
Epoch 62/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 111s 472ms/step - accuracy: 0.9654 - loss: 0.1839 - val_accuracy: 0.8848 - val_loss: 0.5267 - learning_rate: 2.5000e-04
Epoch 63/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 108s 458ms/step - accuracy: 0.9705 - loss: 0.1683 - val_accuracy: 0.8903 - val_loss: 0.5234 - learning_rate: 2.5000e-04
Epoch 64/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 152s 501ms/step - accuracy: 0.9689 - loss: 0.2026 - val_accuracy: 0.8771 - val_loss: 0.5605 - learning_rate: 2.5000e-04
Epoch 65/100
235/235 ━━━━━━━━━━━━━━━━━━━━

235/235 ━━━━━━━━━━━━━━━━━━━━ 129s 548ms/step - accuracy: 0.9706 - loss: 0.1580 - val_accuracy: 0.8983 - val_loss: 0.4955 - learning_rate: 1.2500e-04
Epoch 68/100
104/235 ━━━━━━━━━━━━━━━━━━━━ 1:25 653ms/step - accuracy: 0.9731 - loss: 0.1351

KeyboardInterrupt: 